In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, confusion_matrix, roc_auc_score)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import xgboost as xgb
import re
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("LOTL ATTACK DETECTION: MATHEMATICAL AND EMPIRICAL ANALYSIS")
print("="*80)
print(f"\nConfiguration:")
print(f"  Random Seed: {RANDOM_SEED}")
print(f"  NumPy Version: {np.__version__}")
print(f"  Pandas Version: {pd.__version__}")
# print(f"  Scikit-learn Version: {sklearn.__version__}")

LOTL ATTACK DETECTION: MATHEMATICAL AND EMPIRICAL ANALYSIS

Configuration:
  Random Seed: 42
  NumPy Version: 2.0.2
  Pandas Version: 2.2.2


In [2]:

class MathematicalFramework:
    """
    Mathematical framework for LOTL detection analysis.

    Key concepts:
    1. Distribution shift quantification (KL divergence)
    2. Ensemble error reduction (binomial theorem)
    3. Model diversity (disagreement metric)
    4. Information theory (mutual information)
    """

    @staticmethod
    def kl_divergence_empirical(P, Q, bins=50):
        """
        Compute empirical KL divergence between distributions.

        KL(P||Q) = Σ P(x) log(P(x)/Q(x))

        Args:
            P, Q: Samples from distributions
            bins: Number of bins for histogram estimation

        Returns:
            KL divergence in bits
        """
        # Create histograms
        hist_P, bin_edges = np.histogram(P, bins=bins, density=True)
        hist_Q, _ = np.histogram(Q, bins=bin_edges, density=True)

        # Add small constant to avoid log(0)
        epsilon = 1e-10
        hist_P = hist_P + epsilon
        hist_Q = hist_Q + epsilon

        # Normalize
        hist_P = hist_P / hist_P.sum()
        hist_Q = hist_Q / hist_Q.sum()

        # Compute KL divergence
        kl = np.sum(hist_P * np.log2(hist_P / hist_Q))

        return kl

    @staticmethod
    def ensemble_error_bound(k, epsilon):
        """
        Theoretical ensemble error bound via majority voting.

        Theorem: For k independent classifiers with error ε < 0.5,
        majority voting achieves error:

        P(error) = Σ(i=⌈k/2⌉ to k) C(k,i) * ε^i * (1-ε)^(k-i)

        Args:
            k: Number of classifiers
            epsilon: Individual classifier error rate

        Returns:
            Ensemble error probability
        """
        from scipy.special import comb

        error_prob = 0.0
        for i in range(int(np.ceil(k/2)), k+1):
            error_prob += comb(k, i, exact=True) * (epsilon**i) * ((1-epsilon)**(k-i))

        return error_prob

    @staticmethod
    def model_diversity(predictions_matrix):
        """
        Compute pairwise diversity between models.

        Diversity = (1/(k(k-1))) * Σ(i≠j) [1 - agreement(i,j)]

        Args:
            predictions_matrix: [n_samples, n_models] binary predictions

        Returns:
            Average diversity score
        """
        n_samples, n_models = predictions_matrix.shape

        diversity_sum = 0.0
        count = 0

        for i in range(n_models):
            for j in range(i+1, n_models):
                agreement = np.mean(predictions_matrix[:, i] == predictions_matrix[:, j])
                diversity_sum += (1 - agreement)
                count += 1

        return diversity_sum / count if count > 0 else 0.0

    @staticmethod
    def mutual_information_binary(y_true, y_pred):
        """
        Compute mutual information for binary classification.

        I(Y;Ŷ) = H(Y) - H(Y|Ŷ)

        Args:
            y_true: True labels
            y_pred: Predicted labels

        Returns:
            Mutual information in bits
        """
        from sklearn.metrics import mutual_info_score

        # Compute MI using sklearn (in nats)
        mi_nats = mutual_info_score(y_true, y_pred)

        # Convert to bits
        mi_bits = mi_nats / np.log(2)

        return mi_bits

# Instantiate framework
math_framework = MathematicalFramework()

print("\nMathematical Framework Initialized:")
print("  ✓ KL Divergence computation")
print("  ✓ Ensemble error bounds")
print("  ✓ Model diversity metrics")
print("  ✓ Mutual information calculation")


Mathematical Framework Initialized:
  ✓ KL Divergence computation
  ✓ Ensemble error bounds
  ✓ Model diversity metrics
  ✓ Mutual information calculation


In [7]:
def load_and_preprocess_data(train_path, volt_typhoon_path):
    import pandas as pd

    print("\n[1/3] Loading training data...")
    train_df = pd.read_csv(train_path)

    print("[2/3] Loading Volt Typhoon data...")
    volt_df = pd.read_csv(volt_typhoon_path)

    # Detect column names safely
    def get_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        raise ValueError(f"Required columns {candidates} not found in dataset: {df.columns}")

    command_col = get_col(train_df, ["command", "cmd", "Command", "raw_cmd"])
    label_col   = get_col(train_df, ["label", "Label", "class", "Class", "attack"])

    print("[3/3] Preprocessing...")

    for df in [train_df, volt_df]:
        # Remove nulls
        df.dropna(subset=[command_col, label_col], inplace=True)

        # Convert to string and lowercase
        df["command_lower"] = df[command_col].astype(str).str.lower()

        # Basic cleaning
        df["command_clean"] = df["command_lower"].apply(lambda x: " ".join(x.split()))

        # Convert label to int if needed
        df[label_col] = df[label_col].astype(int)

    # Print dataset stats
    print("\nDataset Statistics:")
    for name, df in [("Training set", train_df), ("Volt Typhoon", volt_df)]:
        print(f"  {name}: {len(df)} samples")
        print(f"    - Benign: {(df[label_col] == 0).sum()}")

    return train_df, volt_df

In [4]:
def extract_command_features(command):
    """
    Extract 20 complexity features from a command.

    Features capture:
    - Syntactic complexity (length, tokens, operators)
    - Encoding/obfuscation indicators
    - Suspicious patterns

    Returns:
        Dict with 20 features
    """
    if pd.isna(command) or not command:
        return {f'feature_{i}': 0.0 for i in range(20)}

    cmd = str(command)
    cmd_lower = cmd.lower()

    features = {}

    # Syntactic complexity
    features['cmd_length'] = min(len(cmd) / 1000.0, 1.0)
    features['token_count'] = min(len(cmd.split()) / 50.0, 1.0)
    features['pipe_count'] = min(cmd.count('|') / 5.0, 1.0)
    features['redirect_count'] = min((cmd.count('>') + cmd.count('<')) / 3.0, 1.0)
    features['semicolon_count'] = min(cmd.count(';') / 3.0, 1.0)
    features['ampersand_count'] = min(cmd.count('&') / 3.0, 1.0)

    # Encoding/obfuscation
    features['has_base64'] = float(bool(re.search(r'[A-Za-z0-9+/]{40,}={0,2}', cmd)))
    features['has_encoding'] = float(bool(re.search(r'-enc|-en|-e\s+', cmd_lower)))
    features['has_hex'] = float(bool(re.search(r'0x[0-9a-f]{4,}', cmd_lower)))
    features['special_char_ratio'] = sum(not c.isalnum() and not c.isspace() for c in cmd) / max(len(cmd), 1)

    # Quotes and nesting
    features['nested_quotes'] = min(cmd.count('"') / 4.0, 1.0)
    features['single_quotes'] = min(cmd.count("'") / 4.0, 1.0)
    features['backtick_count'] = min(cmd.count('`') / 3.0, 1.0)

    # Suspicious patterns
    features['has_network_path'] = float(bool(re.search(r'\\\\[\d\.]+', cmd)))
    features['has_localhost'] = float(bool(re.search(r'(127\.0\.0\.1|localhost)', cmd_lower)))
    features['has_admin_share'] = float(bool(re.search(r'\\\\.*\\(ADMIN|C)\$', cmd, re.IGNORECASE)))
    features['has_powershell'] = float(bool(re.search(r'powershell|pwsh', cmd_lower)))
    features['has_wmi'] = float(bool(re.search(r'wmic|Get-WmiObject', cmd, re.IGNORECASE)))

    # Command chaining
    features['command_chain_length'] = min(
        cmd_lower.count('|') + cmd_lower.count(';') + cmd_lower.count('&'),
        5.0
    ) / 5.0

    # Silent mode
    features['has_quiet_mode'] = float(bool(re.search(r'/[qQ]|-quiet|-silent', cmd)))

    return features

def create_feature_matrix(df):
    """
    Create feature matrix from dataframe.

    Returns:
        X: Feature matrix [n_samples, 20]
        feature_names: List of feature names
    """
    print("\nExtracting features...")
    features_list = []

    for cmd in df['command']:
        features = extract_command_features(cmd)
        features_list.append(list(features.values()))

    X = np.array(features_list)
    feature_names = list(extract_command_features("dummy").keys())

    print(f"  Feature matrix shape: {X.shape}")
    print(f"  Features: {feature_names[:5]}... (20 total)")

    return X, feature_names

print("\nFeature Engineering Functions Defined:")
print("  - extract_command_features(): Extracts 20 complexity features")
print("  - create_feature_matrix(): Converts commands to feature matrix")


SECTION 4: FEATURE ENGINEERING

Feature Engineering Functions Defined:
  - extract_command_features(): Extracts 20 complexity features
  - create_feature_matrix(): Converts commands to feature matrix


In [ ]:
train_df, volt_df = load_and_preprocess_data('lotl_benign_dataset.csv', 'LOTL Spreadsheet Volttyphoon.csv')

print("\n--- Training Data (train_df) --- ")
print(train_df.head())

print("\n--- Volt Typhoon Data (volt_df) --- ")
print(volt_df.head())

In [5]:



# ============================================================================
# SECTION 5: BASELINE MODEL TRAINING
# ============================================================================

print("\n" + "="*80)
print("SECTION 5: BASELINE MODEL TRAINING")
print("="*80)

def train_baseline_models(X_train, y_train, X_val, y_val):
    """
    Train baseline SOTA models.

    Models:
    1. Random Forest + TF-IDF
    2. XGBoost + features
    3. Gradient Boosting

    Returns:
        models: Dict of trained models
        results: Dict of validation results
    """
    print("\n[1/4] Training Random Forest...")
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_val)

    print("[2/4] Training XGBoost...")
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=10,
        learning_rate=0.1,
        random_state=RANDOM_SEED,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    xgb_pred = xgb_model.predict(X_val)

    print("[3/4] Training Gradient Boosting...")
    gb_model = GradientBoostingClassifier(
        n_estimators=100,
        max_depth=10,
        learning_rate=0.1,
        random_state=RANDOM_SEED
    )
    gb_model.fit(X_train, y_train)
    gb_pred = gb_model.predict(X_val)

    print("[4/4] Computing metrics...")

    models = {
        'Random Forest': rf_model,
        'XGBoost': xgb_model,
        'Gradient Boosting': gb_model
    }

    results = {}
    for name, pred in [('Random Forest', rf_pred), ('XGBoost', xgb_pred), ('Gradient Boosting', gb_pred)]:
        results[name] = {
            'accuracy': accuracy_score(y_val, pred),
            'precision': precision_score(y_val, pred, zero_division=0),
            'recall': recall_score(y_val, pred, zero_division=0),
            'f1': f1_score(y_val, pred, zero_division=0)
        }

    # Print results
    print("\nValidation Results:")
    print("-" * 60)
    print(f"{'Model':<20} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
    print("-" * 60)
    for name, metrics in results.items():
        print(f"{name:<20} {metrics['accuracy']:<12.4f} {metrics['precision']:<12.4f} "
              f"{metrics['recall']:<12.4f} {metrics['f1']:<12.4f}")

    return models, results

print("\nBaseline Model Training Function Defined")
print("  Usage: models, results = train_baseline_models(X_train, y_train, X_val, y_val)")


SECTION 5: BASELINE MODEL TRAINING

Baseline Model Training Function Defined
  Usage: models, results = train_baseline_models(X_train, y_train, X_val, y_val)


In [6]:

# ============================================================================
# SECTION 6: DISTRIBUTION SHIFT ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("SECTION 6: DISTRIBUTION SHIFT ANALYSIS")
print("="*80)

def analyze_distribution_shift(train_features, volt_features, feature_names):
    """
    Quantify distribution shift between training and Volt Typhoon data.

    Metrics:
    1. KL Divergence (bits)
    2. Jensen-Shannon Distance
    3. Feature-wise comparison
    4. Statistical significance tests

    Returns:
        shift_metrics: Dict of shift measurements
    """
    print("\n[1/5] Computing KL Divergence...")

    # Compute KL divergence for each feature
    kl_divergences = []
    for i, feature_name in enumerate(feature_names):
        train_values = train_features[:, i]
        volt_values = volt_features[:, i]

        # Skip if no variance
        if train_values.std() < 1e-6 or volt_values.std() < 1e-6:
            kl_divergences.append(0.0)
            continue

        kl = math_framework.kl_divergence_empirical(train_values, volt_values)
        kl_divergences.append(kl)

    avg_kl = np.mean([k for k in kl_divergences if k > 0])

    print(f"  Average KL Divergence: {avg_kl:.3f} bits")

    print("\n[2/5] Computing Jensen-Shannon Distance...")

    # Flatten and compute JS distance
    js_distances = []
    for i in range(train_features.shape[1]):
        train_hist, _ = np.histogram(train_features[:, i], bins=20, density=True)
        volt_hist, _ = np.histogram(volt_features[:, i], bins=20, density=True)

        train_hist = train_hist + 1e-10
        volt_hist = volt_hist + 1e-10

        train_hist /= train_hist.sum()
        volt_hist /= volt_hist.sum()

        js = jensenshannon(train_hist, volt_hist)
        js_distances.append(js)

    avg_js = np.mean(js_distances)
    print(f"  Average JS Distance: {avg_js:.3f}")

    print("\n[3/5] Feature-wise statistical tests...")

    # Kolmogorov-Smirnov test for each feature
    ks_results = []
    for i, feature_name in enumerate(feature_names):
        statistic, pvalue = stats.ks_2samp(train_features[:, i], volt_features[:, i])
        ks_results.append({
            'feature': feature_name,
            'statistic': statistic,
            'pvalue': pvalue,
            'significant': pvalue < 0.05
        })

    n_significant = sum(1 for r in ks_results if r['significant'])
    print(f"  Features with significant shift: {n_significant}/{len(feature_names)}")

    print("\n[4/5] Computing feature distribution overlap...")

    # Compute overlap coefficient
    overlaps = []
    for i in range(train_features.shape[1]):
        train_min, train_max = train_features[:, i].min(), train_features[:, i].max()
        volt_min, volt_max = volt_features[:, i].min(), volt_features[:, i].max()

        overlap_start = max(train_min, volt_min)
        overlap_end = min(train_max, volt_max)

        if overlap_end > overlap_start:
            overlap_range = overlap_end - overlap_start
            total_range = max(train_max, volt_max) - min(train_min, volt_min)
            overlap_coef = overlap_range / total_range if total_range > 0 else 1.0
        else:
            overlap_coef = 0.0

        overlaps.append(overlap_coef)

    avg_overlap = np.mean(overlaps)
    print(f"  Average feature overlap: {avg_overlap:.3f}")

    print("\n[5/5] Estimating novel pattern ratio...")

    # Estimate percentage of Volt Typhoon samples that are "novel"
    # Using distance-based criterion
    from sklearn.neighbors import NearestNeighbors

    nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(train_features)
    distances, _ = nbrs.kneighbors(volt_features)

    # Samples with distance > 95th percentile of training set self-distances
    train_nbrs = NearestNeighbors(n_neighbors=2, algorithm='ball_tree').fit(train_features)
    train_distances, _ = train_nbrs.kneighbors(train_features)
    train_self_dist = train_distances[:, 1]  # Distance to second nearest (not self)
    threshold = np.percentile(train_self_dist, 95)

    novel_count = np.sum(distances.flatten() > threshold)
    novel_ratio = novel_count / len(volt_features)

    print(f"  Novel pattern ratio: {novel_ratio:.3f} ({novel_ratio*100:.1f}%)")

    # Compile results
    shift_metrics = {
        'kl_divergence': avg_kl,
        'js_distance': avg_js,
        'feature_overlap': avg_overlap,
        'novel_ratio': novel_ratio,
        'n_significant_features': n_significant,
        'ks_tests': ks_results,
        'kl_per_feature': dict(zip(feature_names, kl_divergences))
    }

    return shift_metrics

print("\nDistribution Shift Analysis Function Defined")
print("  Usage: shift_metrics = analyze_distribution_shift(train_X, volt_X, feature_names)")


SECTION 6: DISTRIBUTION SHIFT ANALYSIS

Distribution Shift Analysis Function Defined
  Usage: shift_metrics = analyze_distribution_shift(train_X, volt_X, feature_names)


In [7]:

# ============================================================================
# SECTION 7: DYNAMIC ENSEMBLE FRAMEWORK
# ============================================================================

print("\n" + "="*80)
print("SECTION 7: DYNAMIC ENSEMBLE FRAMEWORK")
print("="*80)

class DynamicEnsembleDetector:
    """
    Dynamic ensemble detector with:
    1. High-confidence pattern matching
    2. Adaptive model weighting
    3. Confidence calibration

    Mathematical Foundation:
        Ensemble(x) = argmax_y Σᵢ wᵢ(x) * P(Y=y|x, fᵢ)

    where wᵢ(x) are complexity-adaptive weights.
    """

    def __init__(self, models, pattern_threshold=0.95):
        """
        Initialize ensemble detector.

        Args:
            models: Dict of base models {name: model}
            pattern_threshold: Minimum confidence for pattern detection
        """
        self.models = models
        self.pattern_threshold = pattern_threshold
        self.patterns = self._initialize_patterns()

    def _initialize_patterns(self):
        """Initialize high-confidence attack patterns."""
        return [
            # Pattern, Confidence, Label
            (r'netsh.*portproxy.*v4tov4', 1.0, 1),
            (r'ntdsutil.*snapshot.*create', 1.0, 1),
            (r'reg\s+save\s+hklm\\(sam|system)', 1.0, 1),
            (r'vssadmin.*create.*shadow', 0.95, 1),
            (r'get-eventlog.*security.*4624', 0.90, 1),
            (r'wmic.*process.*call.*create', 0.90, 1),
        ]

    def _check_patterns(self, command):
        """Check if command matches high-confidence patterns."""
        if pd.isna(command):
            return None, 0.0

        cmd_lower = str(command).lower()

        for pattern, confidence, label in self.patterns:
            if re.search(pattern, cmd_lower, re.IGNORECASE):
                return label, confidence

        return None, 0.0

    def predict(self, X_features, commands=None):
        """
        Predict using dynamic ensemble.

        Args:
            X_features: Feature matrix [n_samples, n_features]
            commands: Optional list of command strings for pattern matching

        Returns:
            predictions: Array of predictions
            confidences: Array of confidence scores
            methods: List of detection methods used
        """
        n_samples = X_features.shape[0]
        predictions = np.zeros(n_samples, dtype=int)
        confidences = np.zeros(n_samples)
        methods = []

        for i in range(n_samples):
            # Stage 1: Pattern matching
            if commands is not None:
                pattern_label, pattern_conf = self._check_patterns(commands[i])
                if pattern_label is not None and pattern_conf >= self.pattern_threshold:
                    predictions[i] = pattern_label
                    confidences[i] = pattern_conf
                    methods.append('pattern')
                    continue

            # Stage 2: Ensemble voting
            model_preds = []
            model_probs = []

            for name, model in self.models.items():
                try:
                    if hasattr(model, 'predict_proba'):
                        proba = model.predict_proba(X_features[i:i+1])[0]
                        model_preds.append(int(proba[1] > 0.5))
                        model_probs.append(proba[1])
                    else:
                        pred = model.predict(X_features[i:i+1])[0]
                        model_preds.append(int(pred))
                        model_probs.append(0.8 if pred == 1 else 0.2)
                except:
                    model_preds.append(0)
                    model_probs.append(0.5)

            # Weighted voting (uniform weights for simplicity)
            avg_prob = np.mean(model_probs)
            predictions[i] = int(avg_prob > 0.5)
            confidences[i] = avg_prob
            methods.append('ensemble')

        return predictions, confidences, methods

    def evaluate(self, X_features, y_true, commands=None):
        """
        Evaluate ensemble on test set.

        Returns:
            metrics: Dict of performance metrics
        """
        predictions, confidences, methods = self.predict(X_features, commands)

        # Compute metrics
        metrics = {
            'accuracy': accuracy_score(y_true, predictions),
            'precision': precision_score(y_true, predictions, zero_division=0),
            'recall': recall_score(y_true, predictions, zero_division=0),
            'f1': f1_score(y_true, predictions, zero_division=0),
            'pattern_coverage': sum(1 for m in methods if m == 'pattern') / len(methods),
            'ensemble_coverage': sum(1 for m in methods if m == 'ensemble') / len(methods),
            'avg_confidence': np.mean(confidences)
        }

        return metrics, predictions, confidences

print("\nDynamic Ensemble Detector Class Defined:")
print("  - Pattern-based detection (6 high-confidence patterns)")
print("  - Adaptive ensemble voting")
print("  - Comprehensive evaluation metrics")


SECTION 7: DYNAMIC ENSEMBLE FRAMEWORK

Dynamic Ensemble Detector Class Defined:
  - Pattern-based detection (6 high-confidence patterns)
  - Adaptive ensemble voting
  - Comprehensive evaluation metrics


In [8]:


# ============================================================================
# SECTION 8: COMPREHENSIVE EVALUATION PIPELINE
# ============================================================================

print("\n" + "="*80)
print("SECTION 8: COMPREHENSIVE EVALUATION PIPELINE")
print("="*80)

def run_comprehensive_evaluation(train_df, volt_df, models):
    """
    Run complete evaluation pipeline:
    1. Feature extraction
    2. Baseline evaluation
    3. Distribution shift analysis
    4. Ensemble evaluation
    5. Ablation study
    6. Statistical validation

    Returns:
        results: Dict with all evaluation results
    """
    print("\n" + "="*70)
    print("COMPREHENSIVE EVALUATION PIPELINE")
    print("="*70)

    # Step 1: Feature extraction
    print("\n[Step 1/6] Feature Extraction")
    print("-" * 70)

    X_train, feature_names = create_feature_matrix(train_df)
    X_volt, _ = create_feature_matrix(volt_df)

    y_train = train_df['label'].values
    y_volt = volt_df['label'].values

    print(f"  Training features: {X_train.shape}")
    print(f"  Volt Typhoon features: {X_volt.shape}")

    # Step 2: Baseline evaluation on Volt Typhoon
    print("\n[Step 2/6] Baseline Model Evaluation on Volt Typhoon")
    print("-" * 70)

    baseline_results = {}
    for name, model in models.items():
        pred = model.predict(X_volt)

        baseline_results[name] = {
            'accuracy': accuracy_score(y_volt, pred),
            'precision': precision_score(y_volt, pred, zero_division=0),
            'recall': recall_score(y_volt, pred, zero_division=0),
            'f1': f1_score(y_volt, pred, zero_division=0)
        }

        print(f"\n  {name}:")
        print(f"    Accuracy:  {baseline_results[name]['accuracy']:.4f}")
        print(f"    Precision: {baseline_results[name]['precision']:.4f}")
        print(f"    Recall:    {baseline_results[name]['recall']:.4f}")
        print(f"    F1-Score:  {baseline_results[name]['f1']:.4f}")

    # Step 3: Distribution shift analysis
    print("\n[Step 3/6] Distribution Shift Analysis")
    print("-" * 70)

    shift_metrics = analyze_distribution_shift(X_train, X_volt, feature_names)

    print(f"\n  Distribution Shift Summary:")
    print(f"    KL Divergence:     {shift_metrics['kl_divergence']:.3f} bits")
    print(f"    JS Distance:       {shift_metrics['js_distance']:.3f}")
    print(f"    Feature Overlap:   {shift_metrics['feature_overlap']:.3f}")
    print(f"    Novel Ratio:       {shift_metrics['novel_ratio']:.3f}")
    print(f"    Significant Feats: {shift_metrics['n_significant_features']}/{len(feature_names)}")

    # Step 4: Ensemble evaluation
    print("\n[Step 4/6] Dynamic Ensemble Evaluation")
    print("-" * 70)

    ensemble = DynamicEnsembleDetector(models, pattern_threshold=0.95)

    commands_volt = volt_df['command'].values if 'command' in volt_df.columns else None

    ensemble_metrics, ensemble_preds, ensemble_confs = ensemble.evaluate(
        X_volt, y_volt, commands=commands_volt
    )

    print(f"\n  Dynamic Ensemble Results:")
    print(f"    Accuracy:         {ensemble_metrics['accuracy']:.4f}")
    print(f"    Precision:        {ensemble_metrics['precision']:.4f}")
    print(f"    Recall:           {ensemble_metrics['recall']:.4f}")
    print(f"    F1-Score:         {ensemble_metrics['f1']:.4f}")
    print(f"    Pattern Coverage: {ensemble_metrics['pattern_coverage']:.3f}")
    print(f"    Avg Confidence:   {ensemble_metrics['avg_confidence']:.3f}")

    # Step 5: Ablation study
    print("\n[Step 5/6] Ablation Study")
    print("-" * 70)

    # Simple majority voting (no patterns)
    simple_preds = []
    for i in range(len(X_volt)):
        model_votes = []
        for model in models.values():
            pred = model.predict(X_volt[i:i+1])[0]
            model_votes.append(int(pred))
        simple_preds.append(int(np.mean(model_votes) > 0.5))

    simple_preds = np.array(simple_preds)

    ablation_results = {
        'Full System': ensemble_metrics,
        'Simple Majority': {
            'accuracy': accuracy_score(y_volt, simple_preds),
            'precision': precision_score(y_volt, simple_preds, zero_division=0),
            'recall': recall_score(y_volt, simple_preds, zero_division=0),
            'f1': f1_score(y_volt, simple_preds, zero_division=0)
        }
    }

    # Add best single model
    best_model_name = max(baseline_results.keys(),
                         key=lambda k: baseline_results[k]['accuracy'])
    ablation_results['Best Single Model'] = baseline_results[best_model_name]

    print(f"\n  Component Contribution:")
    print(f"    Best Single Model: {ablation_results['Best Single Model']['accuracy']:.4f}")
    print(f"    Simple Ensemble:   {ablation_results['Simple Majority']['accuracy']:.4f} "
          f"(+{ablation_results['Simple Majority']['accuracy'] - ablation_results['Best Single Model']['accuracy']:.4f})")
    print(f"    Full System:       {ablation_results['Full System']['accuracy']:.4f} "
          f"(+{ablation_results['Full System']['accuracy'] - ablation_results['Simple Majority']['accuracy']:.4f})")

    # Step 6: Statistical validation
    print("\n[Step 6/6] Statistical Validation")
    print("-" * 70)

    # Compute PAC bound
    k = len(models)
    n = len(X_train)
    delta = 0.05
    empirical_error = 1 - ensemble_metrics['accuracy']

    pac_bound = empirical_error + np.sqrt(np.log(k/delta) / (2*n))

    print(f"\n  PAC Learning Bound (δ=0.05):")
    print(f"    Empirical Error: {empirical_error:.4f}")
    print(f"    Error Bound:     {pac_bound:.4f}")
    print(f"    Accuracy Bound:  {1 - pac_bound:.4f}")
    print(f"    Status:          {'✓ Within bound' if empirical_error <= pac_bound else '✗ Exceeds bound'}")

    # Compute ensemble theoretical bound
    avg_error = np.mean([1 - r['accuracy'] for r in baseline_results.values()])
    theoretical_ensemble_error = math_framework.ensemble_error_bound(k, avg_error)

    print(f"\n  Ensemble Error Bound:")
    print(f"    Avg Single Error:   {avg_error:.4f}")
    print(f"    Theoretical Bound:  {theoretical_ensemble_error:.4f}")
    print(f"    Actual Error:       {empirical_error:.4f}")
    print(f"    Improvement:        {(theoretical_ensemble_error - empirical_error):.4f}")

    # Compute model diversity
    pred_matrix = np.array([model.predict(X_volt) for model in models.values()]).T
    diversity = math_framework.model_diversity(pred_matrix)

    print(f"\n  Model Diversity:")
    print(f"    Pairwise Diversity: {diversity:.3f}")
    print(f"    Interpretation:     {'Good' if diversity > 0.2 else 'Low'} ensemble diversity")

    # Compute mutual information
    mi_values = []
    for name, model in models.items():
        pred = model.predict(X_volt)
        mi = math_framework.mutual_information_binary(y_volt, pred)
        mi_values.append(mi)
        print(f"    MI({name}): {mi:.3f} bits")

    ensemble_mi = math_framework.mutual_information_binary(y_volt, ensemble_preds)
    print(f"    MI(Ensemble): {ensemble_mi:.3f} bits")
    print(f"    Improvement:  {ensemble_mi - max(mi_values):.3f} bits "
          f"({(ensemble_mi - max(mi_values))/max(mi_values)*100:.1f}%)")

    # Compile all results
    results = {
        'baseline_results': baseline_results,
        'shift_metrics': shift_metrics,
        'ensemble_metrics': ensemble_metrics,
        'ablation_results': ablation_results,
        'statistical_validation': {
            'pac_bound': pac_bound,
            'theoretical_ensemble_error': theoretical_ensemble_error,
            'diversity': diversity,
            'mutual_information': {
                'single_models': dict(zip(models.keys(), mi_values)),
                'ensemble': ensemble_mi
            }
        }
    }

    return results

print("\nComprehensive Evaluation Pipeline Defined")
print("  Usage: results = run_comprehensive_evaluation(train_df, volt_df, models)")




SECTION 8: COMPREHENSIVE EVALUATION PIPELINE

Comprehensive Evaluation Pipeline Defined
  Usage: results = run_comprehensive_evaluation(train_df, volt_df, models)


In [9]:
# ============================================================================
# SECTION 9: VISUALIZATION AND REPORTING
# ============================================================================

print("\n" + "="*80)
print("SECTION 9: VISUALIZATION AND REPORTING")
print("="*80)

def create_comprehensive_report(results):
    """
    Create visualizations and summary report.

    Generates:
    1. Performance comparison table
    2. Distribution shift visualization
    3. Ablation study chart
    4. Statistical validation summary
    """
    print("\nGenerating Comprehensive Report...")

    # Create figure with subplots
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

    # Subplot 1: Performance Comparison
    ax1 = fig.add_subplot(gs[0, 0])

    models_list = list(results['baseline_results'].keys()) + ['Ensemble']
    accuracies = [results['baseline_results'][m]['accuracy'] for m in results['baseline_results'].keys()]
    accuracies.append(results['ensemble_metrics']['accuracy'])

    colors = ['#FF6B6B' if acc < 0.7 else '#4ECDC4' if acc < 0.8 else '#95E1D3' for acc in accuracies]

    bars = ax1.barh(models_list, accuracies, color=colors, edgecolor='black', linewidth=1.5)
    ax1.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
    ax1.set_title('Model Performance on Volt Typhoon', fontsize=14, fontweight='bold')
    ax1.set_xlim([0, 1])
    ax1.grid(axis='x', alpha=0.3)

    # Add value labels
    for i, (bar, acc) in enumerate(zip(bars, accuracies)):
        ax1.text(acc + 0.02, i, f'{acc:.3f}', va='center', fontsize=10, fontweight='bold')

    # Subplot 2: Distribution Shift Metrics
    ax2 = fig.add_subplot(gs[0, 1])

    shift_data = {
        'KL Divergence\n(bits)': results['shift_metrics']['kl_divergence'],
        'JS Distance': results['shift_metrics']['js_distance'],
        'Feature\nOverlap': results['shift_metrics']['feature_overlap'],
        'Novel\nRatio': results['shift_metrics']['novel_ratio']
    }

    bars = ax2.bar(range(len(shift_data)), list(shift_data.values()),
                   color=['#FF6B6B', '#FFD93D', '#6BCF7F', '#4D96FF'],
                   edgecolor='black', linewidth=1.5)
    ax2.set_xticks(range(len(shift_data)))
    ax2.set_xticklabels(list(shift_data.keys()), fontsize=9)
    ax2.set_ylabel('Value', fontsize=12, fontweight='bold')
    ax2.set_title('Distribution Shift Analysis', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, shift_data.values())):
        ax2.text(i, val + 0.05, f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

    # Subplot 3: Ablation Study
    ax3 = fig.add_subplot(gs[1, :])

    ablation_configs = list(results['ablation_results'].keys())
    metrics = ['accuracy', 'precision', 'recall', 'f1']

    x = np.arange(len(metrics))
    width = 0.25

    for i, config in enumerate(ablation_configs):
        values = [results['ablation_results'][config][m] for m in metrics]
        ax3.bar(x + i*width, values, width, label=config,
                edgecolor='black', linewidth=1.2)

    ax3.set_xlabel('Metrics', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax3.set_title('Ablation Study: Component Contributions', fontsize=14, fontweight='bold')
    ax3.set_xticks(x + width)
    ax3.set_xticklabels([m.capitalize() for m in metrics])
    ax3.legend(fontsize=10)
    ax3.grid(axis='y', alpha=0.3)
    ax3.set_ylim([0, 1.05])

    # Subplot 4: Statistical Validation
    ax4 = fig.add_subplot(gs[2, 0])

    stat_data = {
        'PAC\nBound': results['statistical_validation']['pac_bound'],
        'Theoretical\nEnsemble\nError': results['statistical_validation']['theoretical_ensemble_error'],
        'Actual\nError': 1 - results['ensemble_metrics']['accuracy']
    }

    bars = ax4.bar(range(len(stat_data)), list(stat_data.values()),
                   color=['#FF6B6B', '#FFD93D', '#95E1D3'],
                   edgecolor='black', linewidth=1.5)
    ax4.set_xticks(range(len(stat_data)))
    ax4.set_xticklabels(list(stat_data.keys()), fontsize=9)
    ax4.set_ylabel('Error Rate', fontsize=12, fontweight='bold')
    ax4.set_title('Theoretical Validation', fontsize=14, fontweight='bold')
    ax4.grid(axis='y', alpha=0.3)

    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, stat_data.values())):
        ax4.text(i, val + 0.01, f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')

    # Subplot 5: Mutual Information
    ax5 = fig.add_subplot(gs[2, 1])

    mi_single = list(results['statistical_validation']['mutual_information']['single_models'].values())
    mi_ensemble = results['statistical_validation']['mutual_information']['ensemble']
    mi_names = list(results['statistical_validation']['mutual_information']['single_models'].keys()) + ['Ensemble']
    mi_values = mi_single + [mi_ensemble]

    bars = ax5.barh(mi_names, mi_values,
                    color=['#FF6B6B']*len(mi_single) + ['#95E1D3'],
                    edgecolor='black', linewidth=1.5)
    ax5.set_xlabel('Mutual Information (bits)', fontsize=12, fontweight='bold')
    ax5.set_title('Information Capture Analysis', fontsize=14, fontweight='bold')
    ax5.grid(axis='x', alpha=0.3)

    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, mi_values)):
        ax5.text(val + 0.02, i, f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig('lotl_detection_comprehensive_analysis.png', dpi=300, bbox_inches='tight')
    print("  ✓ Saved: lotl_detection_comprehensive_analysis.png")

    # Print summary table
    print("\n" + "="*80)
    print("COMPREHENSIVE RESULTS SUMMARY")
    print("="*80)

    print("\n1. BASELINE MODEL PERFORMANCE ON VOLT TYPHOON")
    print("-" * 80)
    print(f"{'Model':<25} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
    print("-" * 80)
    for name, metrics in results['baseline_results'].items():
        print(f"{name:<25} {metrics['accuracy']:<12.4f} {metrics['precision']:<12.4f} "
              f"{metrics['recall']:<12.4f} {metrics['f1']:<12.4f}")

    print("\n2. DISTRIBUTION SHIFT METRICS")
    print("-" * 80)
    print(f"  KL Divergence:          {results['shift_metrics']['kl_divergence']:.4f} bits")
    print(f"  Jensen-Shannon Distance: {results['shift_metrics']['js_distance']:.4f}")
    print(f"  Feature Overlap:         {results['shift_metrics']['feature_overlap']:.4f}")
    print(f"  Novel Pattern Ratio:     {results['shift_metrics']['novel_ratio']:.4f} "
          f"({results['shift_metrics']['novel_ratio']*100:.1f}%)")
    print(f"  Significant Features:    {results['shift_metrics']['n_significant_features']}/20")

    print("\n3. DYNAMIC ENSEMBLE PERFORMANCE")
    print("-" * 80)
    ens = results['ensemble_metrics']
    print(f"  Accuracy:          {ens['accuracy']:.4f}")
    print(f"  Precision:         {ens['precision']:.4f}")
    print(f"  Recall:            {ens['recall']:.4f}")
    print(f"  F1-Score:          {ens['f1']:.4f}")
    print(f"  Pattern Coverage:  {ens['pattern_coverage']:.4f}")
    print(f"  Average Confidence: {ens['avg_confidence']:.4f}")

    print("\n4. ABLATION STUDY")
    print("-" * 80)
    for config, metrics in results['ablation_results'].items():
        print(f"  {config:<25} Accuracy: {metrics['accuracy']:.4f}")

    baseline_acc = results['ablation_results']['Best Single Model']['accuracy']
    ensemble_acc = results['ablation_results']['Full System']['accuracy']
    improvement = ensemble_acc - baseline_acc
    error_reduction = improvement / (1 - baseline_acc)

    print(f"\n  Absolute Improvement: {improvement:.4f} ({improvement*100:.1f}%)")
    print(f"  Error Reduction:      {error_reduction:.4f} ({error_reduction*100:.1f}%)")

    print("\n5. THEORETICAL VALIDATION")
    print("-" * 80)
    val = results['statistical_validation']
    print(f"  PAC Learning Bound:")
    print(f"    Error Bound:     {val['pac_bound']:.4f}")
    print(f"    Actual Error:    {1 - results['ensemble_metrics']['accuracy']:.4f}")
    print(f"    Status:          {'✓ Within bound' if 1 - results['ensemble_metrics']['accuracy'] <= val['pac_bound'] else '✗ Exceeds bound'}")

    print(f"\n  Ensemble Error Bound:")
    print(f"    Theoretical:     {val['theoretical_ensemble_error']:.4f}")
    print(f"    Actual:          {1 - results['ensemble_metrics']['accuracy']:.4f}")
    print(f"    Improvement:     {val['theoretical_ensemble_error'] - (1 - results['ensemble_metrics']['accuracy']):.4f}")

    print(f"\n  Model Diversity:    {val['diversity']:.4f}")
    print(f"  Ensemble MI:        {val['mutual_information']['ensemble']:.4f} bits")
    print(f"  Best Single MI:     {max(val['mutual_information']['single_models'].values()):.4f} bits")
    print(f"  Information Gain:   {val['mutual_information']['ensemble'] - max(val['mutual_information']['single_models'].values()):.4f} bits")

    print("\n" + "="*80)
    print("ANALYSIS COMPLETE")
    print("="*80)

print("\nVisualization and Reporting Function Defined")
print("  Usage: create_comprehensive_report(results)")

# ============================================================================
# SECTION 10: MAIN EXECUTION
# ============================================================================

print("\n" + "="*80)
print("SECTION 10: MAIN EXECUTION")
print("="*80)

print("\n" + "="*80)
print("NOTEBOOK READY FOR EXECUTION")
print("="*80)

print("\nTo run the complete analysis, execute:")
print("""
# Step 1: Load data
train_df, volt_df = load_and_preprocess_data('training_data.csv', 'volt_typhoon.csv')

# Step 2: Extract features
X_train, feature_names = create_feature_matrix(train_df)
y_train = train_df['label'].values

# Step 3: Train-validation split
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_SEED, stratify=y_train
)

# Step 4: Train baseline models
models, train_results = train_baseline_models(X_train_split, y_train_split, X_val_split, y_val_split)

# Step 5: Run comprehensive evaluation
results = run_comprehensive_evaluation(train_df, volt_df, models)

# Step 6: Generate report and visualizations
create_comprehensive_report(results)
""")

print("\n" + "="*80)
print("All functions loaded successfully!")
print("="*80)

print("\nKey Features:")
print("  ✓ Mathematical framework with KL divergence, PAC bounds, MI")
print("  ✓ Comprehensive feature engineering (20 features)")
print("  ✓ Multiple baseline models (RF, XGBoost, GB)")
print("  ✓ Distribution shift analysis")
print("  ✓ Dynamic ensemble with pattern matching")
print("  ✓ Ablation studies")
print("  ✓ Statistical validation")
print("  ✓ Comprehensive visualization")

print("\nExpected Output:")
print("  - Performance comparison tables")
print("  - Distribution shift metrics")
print("  - Ablation study results")
print("  - Theoretical validation")
print("  - Comprehensive visualization (PNG)")

print("\n" + "="*80)
print("Ready to analyze LOTL detection on Volt Typhoon! 🚀")
print("="*80)


SECTION 9: VISUALIZATION AND REPORTING

Visualization and Reporting Function Defined
  Usage: create_comprehensive_report(results)

SECTION 10: MAIN EXECUTION

NOTEBOOK READY FOR EXECUTION

To run the complete analysis, execute:

# Step 1: Load data
train_df, volt_df = load_and_preprocess_data('training_data.csv', 'volt_typhoon.csv')

# Step 2: Extract features
X_train, feature_names = create_feature_matrix(train_df)
y_train = train_df['label'].values

# Step 3: Train-validation split
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_SEED, stratify=y_train
)

# Step 4: Train baseline models
models, train_results = train_baseline_models(X_train_split, y_train_split, X_val_split, y_val_split)

# Step 5: Run comprehensive evaluation
results = run_comprehensive_evaluation(train_df, volt_df, models)

# Step 6: Generate report and visualizations
create_comprehensive_report(results)


All functions loaded succe